# Ngày 27 · Embedding, attention và Transformer

- Cài scaled dot-product attention và mặt nạ nhân quả (causal mask)
- Tạo positional encoding hình sin
- Ghép nhiều đầu (multi-head) từ trọng số của nn.MultiheadAttention

**Đọc:**
- Vaswani et al. (2017), Attention Is All You Need, NeurIPS 30
- Zhang, Lipton, Li & Smola, Dive into Deep Learning, chương Attention Mechanisms and Transformers
- Tài liệu PyTorch, torch.nn.MultiheadAttention và torch.nn.functional.scaled_dot_product_attention

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
try:
    import torch
    from torch import nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError('Kernel này chưa có PyTorch. Cài bằng Python của Jupyter: python -m pip install -r requirements-py30.txt') from error
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
# load_digits (UCI Optical Recognition of Handwritten Digits): 1797 ảnh 8×8, chia 80/20 phân tầng, seed 0.
_Xd, _yd = load_digits(return_X_y=True)
_itr, _iva = train_test_split(np.arange(len(_yd)), test_size=0.2, random_state=0, stratify=_yd)
X_tr = torch.tensor(_Xd[_itr] / 16, dtype=torch.float32); y_tr = torch.tensor(_yd[_itr])
X_va = torch.tensor(_Xd[_iva] / 16, dtype=torch.float32); y_va = torch.tensor(_yd[_iva])
def _loader(X, y, batch_size, seed):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed))
def _train_epoch(model, loader, loss_fn, optimizer):
    model.train(); total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); optimizer.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n
def _accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())
torch.manual_seed(27)
_q27, _k27, _v27 = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
_mha27 = nn.MultiheadAttention(embed_dim=12, num_heads=3, batch_first=True)
_x27 = torch.randn(2, 6, 12)


## Scaled dot-product attention, mặt nạ và positional encoding

Gợi ý: scores = q @ k.transpose(−2, −1) / √d; nếu có mask (bool, True = được nhìn) thì scores.masked_fill(~mask, −inf); weights = scores.softmax(−1); out = weights @ v. causal_mask: torch.tril(torch.ones(T, T, dtype=torch.bool)).

In [ ]:
def attention(q, k, v, mask=None):
    # Trả (out, weights).
    raise NotImplementedError

def causal_mask(T):
    raise NotImplementedError

def positional_encoding(T, d):
    # Tensor (T, d); d chẵn.
    raise NotImplementedError


In [ ]:
_report('d27-attention',
    _check('Khớp F.scaled_dot_product_attention', lambda: (torch.allclose(attention(_q27, _k27, _v27)[0], F.scaled_dot_product_attention(_q27, _k27, _v27), atol=1e-5), 'Không mặt nạ.')),
    _check('Mặt nạ nhân quả: khớp is_causal=True và không nhìn tương lai', lambda: ((lambda r: torch.allclose(r[0], F.scaled_dot_product_attention(_q27, _k27, _v27, is_causal=True), atol=1e-5) and float(r[1].triu(1).abs().sum()) == 0)(attention(_q27, _k27, _v27, causal_mask(5))), 'Trọng số phía trên đường chéo bằng 0.')),
    _check('Positional encoding đúng công thức Vaswani et al.', lambda: ((lambda pe: tuple(pe.shape) == (50, 16) and torch.allclose(pe[0], torch.tensor([0., 1.] * 8)) and np.isclose(float(pe[7, 4]), math.sin(7 / 10000 ** (4 / 16)), atol=1e-6) and np.isclose(float(pe[7, 5]), math.cos(7 / 10000 ** (4 / 16)), atol=1e-6))(positional_encoding(50, 16)), 'Kiểm tra PE[0] và PE[7, 4:6].')),
)


## Multi-head attention từ trọng số có sẵn

Gợi ý: W_q, W_k, W_v = mha.in_proj_weight.chunk(3) (bias tương tự). Chiếu x, tách thành h đầu: (B, T, E) → (B, h, T, E/h). Dùng attention của bài trên cho từng đầu, ghép lại (B, T, E), rồi qua mha.out_proj.

In [ ]:
def multi_head_attention(x, mha):
    # x: (B, T, E); mha: nn.MultiheadAttention(batch_first=True). Trả tensor (B, T, E) bằng phép tính thủ công.
    raise NotImplementedError


In [ ]:
def _ref_attention(q, k, v, mask=None):
    s = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    return s.softmax(-1) @ v, s.softmax(-1)
attention = globals().get('attention') or _ref_attention
_report('d27-mha',
    _check('Khớp nn.MultiheadAttention (3 đầu, E = 12)', lambda: (torch.allclose(multi_head_attention(_x27, _mha27), _mha27(_x27, _x27, _x27, need_weights=False)[0], atol=1e-5), 'Self-attention, batch_first=True.')),
    _check('Đúng kích thước đầu ra', lambda: (tuple(multi_head_attention(_x27, _mha27).shape) == (2, 6, 12), '(B, T, E).')),
)


## Dự án

Mở notebook cloud Ngay_27_Transformer (Kaggle hoặc Colab) với bộ IMDB 50k review. Huấn luyện mô hình embedding trung bình (bag of embeddings) rồi Transformer encoder nhỏ, so accuracy và thời gian một epoch trên GPU. Ghi độ dài chuỗi tối đa bạn chọn và ảnh hưởng của nó.